# **Lab 4**<br>

We have already implemented convolution, inverse convolution, DFT, and inverse DFT in our previous labs. In today's lab, we will be implementing convolution and inverse convolution with the help of frequency domain.

Complete the following functions from the previous lab tasks:

In [ ]:
import numpy as np

def convInputSide(x, h):
  n = len(x)
  m = len(h)

  y = [0] * (n+m-1)

  for i in range(n+m-1):
    for j in range(n):
      k = i - j
      if(0 <= k < m):
        y[i] += int(x[j]) * int(h[k])
  return y


def build_conv_matrix(h, x_len):
  N = len(h)
  M = N + x_len - 1
  H = np.zeros((M, x_len))

  for i in range(M):
    for j in range(x_len):
      if 0 <= i - j < N:
        H[i, j] = h[i-j]
  return H

def inverse_conv(y,h):
  h_len = len(h)
  x_len = len(y) - len(h) + 1

  H = build_conv_matrix(h, x_len)
  x = np.dot(np.linalg.pinv(H), y)

  return x


def forwardDFT(x,N):
  ReX = np.zeros(int(N/2) + 1)
  ImX = np.zeros(int(N/2) + 1)

  for k in range(int(N/2) +1):
    for n in range(len(x)):
      angle = -2 * np.pi * k * n / N
      ReX[k] += x[n] * np.cos(angle)
      ImX[k] += x[n] * np.sin(angle)

  return ReX, ImX

def inverseDFT(ReX,ImX):
  N_prime = (len(ReX) - 1) * 2

  for k in range(len(ReX)):
    ReX[k] = ReX[k] / (N_prime / 2)
    ImX[k] = -ImX[k] / (N_prime / 2)

  ReX[0] = ReX[0] / 2
  ReX[-1] = ReX[- 1] / 2

  xx = np.zeros(N_prime)

  for i in range(N_prime):
    for k in range(len(ReX)):
      angle = 2 * np.pi * k * i / N_prime

      xx[i] += ReX[k] * np.cos(angle)
      xx[i] += ImX[k] * np.sin(angle)

  return xx

##Task 1
Write a function named convertToPolar() that takes the frequency domain components in rectangular form $ReX$ and $ImX$ and returns the polar form of them.


In [ ]:
def convertToPolar(ReX, ImX):
  PhaseX = np.arctan2(ImX, ReX)
  MagX = np.sqrt(ReX**2 + ImX**2)
  return MagX, PhaseX

##Task 2
Write a function named convertToRec() that takes the frequency domain components in polar form $MagX$ and $PhaseX$ and returns the rectangular form of them.

In [ ]:
def convertToRec(MagX, PhaseX):
  ReX = MagX * np.cos(PhaseX)
  ImX = MagX * np.sin(PhaseX)
  return ReX, ImX

## Task 3
Complete the functions named multiplyRec() and multiplyPolar() that take the frequency domain components (in rectangular and polar forms) and multiply them.

In [ ]:
def multiplyRec(ReX, ImX, ReY, ImY):
  #your code here
  Rez = ReX * ReY - ImX * ImY
  Imz = ReX * ImY + ReY * ImX
  return Rez, Imz

def multiplyPolar(MagX, PhaseX, MagY, PhaseY):
  #your code here
  Magz = MagX * MagY
  Phasez = PhaseX + PhaseY
  return Magz, Phasez

## Task 4
Complete the functions named divideRec() and dividePolar() that takes two frequency domain spectra (in rectangular and polar forms) and divides them.

In [ ]:
def divideRec(ReX, ImX, ReY, ImY):
  #your code here
  Rez = (ReX * ReY + ImX * ImY) / (ReX**2 + ImX**2)
  Imz = (ImY * ReX - ReY * ImX) / (ReX**2 + ImX**2)
  return Rez, Imz

def dividePolar(MagX, PhaseX, MagY, PhaseY):
  #your code here
  Magz = MagY / MagX
  PhaseZ = PhaseY - PhaseX
  return Magz, PhaseZ

## Task 5
Use the functions above to find the output $y$ of the given input signal $x$ when passed through a system with impulse response $h$. You have to show that convolution in time domain gives the same result when compared to the output you get by performing forward DFT, multiplication in frequency domain, and inverse DFT. You need to show it for both the frequency domain notations.

In [ ]:
x = np.array([-5, 3, -1, 4, 5])
h = np.array([-4, 1, -2, 3])
#your code here

y = convInputSide(x, h)
print(y)

N = len(x) + len(h) - 1
x_padded = np.pad(x, (0, N - len(x)))
h_padded = np.pad(h, (0, N - len(h)))

ReX, ImX = forwardDFT(x_padded, len(x_padded))
MagX, PhaseX = convertToPolar(ReX, ImX)

ReH, ImH = forwardDFT(h_padded, len(h_padded))
MagH, PhaseH = convertToPolar(ReH, ImH)

ReY, ImY = multiplyRec(ReX, ImX, ReH, ImH)
y1 = inverseDFT(ReY, ImY)
print(y1)

MagY, PhaseY = multiplyPolar(MagX, PhaseX, MagH, PhaseH)
ReY, ImY = convertToRec(MagY, PhaseY)
y2 = inverseDFT(ReY, ImY)

print(y2)

[20, -17, 17, -38, -5, -6, 2, 15]
[ 20. -17.  17. -38.  -5.  -6.   2.  15.]
[ 20. -17.  17. -38.  -5.  -6.   2.  15.]


## Task 6
Now you are given the output $y$ and the input $x$. You need to show the same as Task 3, but for inverse convolution.

In [ ]:
x = np.array([-5, 3, -1, 4, 5])
y = np.array([-10, 1, -9, 38, -3, 18, -10, -25])
#your code here

h = inverse_conv(y, x)
print(h)

N = len(y)
x_padded = np.pad(x, (0, N - len(x)))

ReX, ImX = forwardDFT(x_padded, len(x_padded))
MagX, PhaseX = convertToPolar(ReX, ImX)

ReY, ImY = forwardDFT(y, len(y))
MagY, PhaseY = convertToPolar(ReY, ImY)

ReH, ImH = divideRec(ReX, ImX, ReY, ImY)
h1 = inverseDFT(ReH, ImH)
print(h1[:len(x)-1])

MagH, PhaseH = dividePolar(MagX, PhaseX, MagY, PhaseY)
ReH, ImH = convertToRec(MagH, PhaseH)
h2 = inverseDFT(ReH, ImH)
print(h2[:len(x)-1])

[ 2.  1.  2. -5.]
[ 2.  1.  2. -5.]
[ 2.  1.  2. -5.]
